In [1]:
# Librerías
import requests
import selectolax
from selectolax.parser import HTMLParser
import pandas as pd
import numpy as np
from datetime import datetime
from zoneinfo import ZoneInfo

# Funciones del proyecto
import scrapping_functions
from scrapping_functions import get_json_from_url, save_json, parse_json_to_model
import html_utils
from html_utils import esperar, obtener_max_page, extraer_productos, save_df_as_csv

##### Categorias

In [2]:
response = requests.get("https://www.stock.com.py/default.aspx")
html = response.text
tree = HTMLParser(html)

In [3]:
data = []

# Buscar todos los nodos de nivel 1
for lvl1_li in tree.css("li.level1"):
    lvl1_a = lvl1_li.css_first("a")
    if not lvl1_a:
        continue
    lvl1_name = lvl1_a.text(strip=True)

    # Dentro de este <li>, buscar los hijos de nivel 2
    for lvl2_li in lvl1_li.css("ul > li.level2"):
        lvl2_a = lvl2_li.css_first("a")
        if not lvl2_a:
            continue
        lvl2_name = lvl2_a.text(strip=True)

        # Dentro del lvl2, buscar los hijos de nivel 3 (con enlaces)
        for lvl3_li in lvl2_li.css("ul > li.level3"):
            lvl3_a = lvl3_li.css_first("a[href]")
            if not lvl3_a:
                continue
            lvl3_name = lvl3_a.text(strip=True)
            lvl3_url = lvl3_a.attributes.get("href")

            data.append({
                "categoria_nivel_1": lvl1_name,
                "categoria_nivel_2": lvl2_name,
                "categoria_nivel_3": lvl3_name,
                "url": lvl3_url,
                "category_slug": f"{lvl1_name}/{lvl2_name}/{lvl3_name}".replace(" ", "_")
            })

In [4]:
df_categorias = pd.DataFrame(data)
df_categorias.nunique()

categoria_nivel_1     19
categoria_nivel_2     81
categoria_nivel_3    293
url                  309
category_slug        309
dtype: int64

In [5]:
df_categorias.head()

,categoria_nivel_1,categoria_nivel_2,categoria_nivel_3,url,category_slug
0,Almacén,Aderezos/Condimentos,Aceites,https://www.stock.com.py/category/3-almacen-ad...,Almacén/Aderezos/Condimentos/Aceites
1,Almacén,Aderezos/Condimentos,Aderezos/Salsas,https://www.stock.com.py/category/4-almacen-ad...,Almacén/Aderezos/Condimentos/Aderezos/Salsas
2,Almacén,Aderezos/Condimentos,Especias,https://www.stock.com.py/category/5-almacen-ad...,Almacén/Aderezos/Condimentos/Especias
3,Almacén,Aderezos/Condimentos,Ketchup,https://www.stock.com.py/category/6-almacen-ad...,Almacén/Aderezos/Condimentos/Ketchup
4,Almacén,Aderezos/Condimentos,Mayonesa,https://www.stock.com.py/category/7-almacen-ad...,Almacén/Aderezos/Condimentos/Mayonesa


In [6]:
save_df_as_csv(
    dataframe = df_categorias,
    name = 'stock_categorias',
    subfolder = 'stock/categorias'
)

[💾] Guardado en: /workspaces/tesis-ivan-gennaro/scripts/bronze/outputs/stock/categorias/stock_categorias_2025-07-21_23-11-25.csv


# Productos

In [8]:
INGESTION_TIME = datetime.now(ZoneInfo("America/Asuncion"))
SUPERMERCADO = "Stock"
productos_final = []
selectors = {
    "producto": "div.producto",
    "titulo": "h2.product-title",
    "marca": "div.product-brand",
    "precio": "span.price-label",
    "unidad_medida": "span.unidad-medida",
    }

# 🔁 Iterar sobre cada categoría (nivel 3) con contexto
for _, row in df_categorias.iterrows():
    categoria_url = row["url"]
    category_slug = row["category_slug"]

    print(f"\n🔎 Scrapeando categoría: {category_slug}")
    esperar()
    
    response = requests.get(categoria_url)
    tree = HTMLParser(response.text)
    max_page = obtener_max_page(tree)
    print(f"📄 Total de páginas: {max_page}")

    for page in range(1, max_page + 1):
        page_url = f"{categoria_url}?pageindex={page}"
        print(f"➡️ Página {page}: {page_url}")
        esperar()

        resp = requests.get(page_url)
        html_tree = HTMLParser(resp.text)

        contexto = {
            "category_slug": category_slug,
            "ingestion_time": INGESTION_TIME,
            "supermercado": SUPERMERCADO
        }

        productos_pagina = extraer_productos(html_tree, contexto, selectors=selectors)
        productos_final.extend(productos_pagina)


🔎 Scrapeando categoría: Almacén/Aderezos/Condimentos/Aceites
📄 Total de páginas: 3
➡️ Página 1: https://www.stock.com.py/category/3-almacen-aderezoscondimentos-aceites.aspx?pageindex=1
➡️ Página 2: https://www.stock.com.py/category/3-almacen-aderezoscondimentos-aceites.aspx?pageindex=2
➡️ Página 3: https://www.stock.com.py/category/3-almacen-aderezoscondimentos-aceites.aspx?pageindex=3


In [ ]:
df = pd.DataFrame(productos_final)
df.head(10)

,titulo,marca,precio,unidad_medida,category_slug,ingestion_time,supermercado
0,ACEITE DE OLIVA VIRGEN BOTELLA 500,BORGES,96.000,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-21 23:11:25.245602-03:00,Stock
1,DACOLONIA OLEO DE COCO EXTRA VIRGEN 200 ML,,36.400,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-21 23:11:25.245602-03:00,Stock
2,ACEITE BORGES OLIVA BOTELLA 125ML,BORGES,25.350,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-21 23:11:25.245602-03:00,Stock
3,ACEITE BORGES OLIVA EX/VIRG BOTELLA 125ML,BORGES,32.300,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-21 23:11:25.245602-03:00,Stock
4,ACEITE DE OLIVA VIRGEN BOTELLA 250,BORGES,46.550,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-21 23:11:25.245602-03:00,Stock
5,ACEITE DE GIRASOL MIRASOL 500ML,MIRASOL,9.500,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-21 23:11:25.245602-03:00,Stock
6,ACEITE DE CANOLA GOURMET OK 500ML,,14.400,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-21 23:11:25.245602-03:00,Stock
7,OK ACEITE DE GIRASOL 200 ML,,5.450,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-21 23:11:25.245602-03:00,Stock
8,ACEITE DE GIRASOL 900 ML OK,,16.200,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-21 23:11:25.245602-03:00,Stock
9,ACEITE DE SOJA OK 1.5L,,23.700,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-21 23:11:25.245602-03:00,Stock


In [10]:
save_df_as_csv(
    dataframe = df,
    name = 'stock_productos',
    subfolder = 'stock/productos'
)

[💾] Guardado en: /workspaces/tesis-ivan-gennaro/scripts/bronze/outputs/stock/productos/stock_productos_2025-07-21_23-11-53.csv
